 <img style="float: right;" src="img/python.png"><br>



# Kurze Einführung in SQL

SQL (Structured Query Language) ist eine standardisierte Sprache zur Verwaltung und Abfrage von Daten in relationalen Datenbanken. Mit SQL lassen sich Datenstrukturen definieren (Data Definition Language, DDL), Daten manipulieren (Data Manipulation Language, DML) sowie Zugriffs- und Berechtigungsrechte steuern (Data Control Language, DCL). Die Kernkonzepte von SQL basieren auf Tabellen, in denen Daten in Zeilen und Spalten organisiert werden. Beziehungen zwischen Tabellen werden durch Schlüssel (Keys) hergestellt.

# Verbindung zu einer MySQL (MariaDB) Datenbank
Als Erstes muss eine Verbindung zu einer Datenbank hergestellt werden. Dazu müssen 2 neue Bibliotheken der Python-Umgebung hinzugefügt werden:
- mysql
- mysql-connector-python

## Wichtige Datentypen in SQL

### Numerische Datentypen
- **TINYINT**: 1 Byte, Wertebereich -128 bis 127 oder 0 bis 255 (unsigned).
- **SMALLINT**: 2 Bytes, Wertebereich -32.768 bis 32.767.
- **INT** (INTEGER): 4 Bytes, Wertebereich -2.147.483.648 bis 2.147.483.647.
- **BIGINT**: 8 Bytes, Wertebereich -9.223.372.036.854.775.808 bis 9.223.372.036.854.775.807.
- **FLOAT**(p): Gleitkommazahl mit einfacher Genauigkeit (typisch 4 Bytes, etwa 7 Dezimalstellen Genauigkeit).
- **DOUBLE**: Gleitkommazahl mit doppelter Genauigkeit (8 Bytes, etwa 15 Dezimalstellen Genauigkeit).
  - **Hinweis**: FLOAT und DOUBLE nutzen binäre Gleitkommadarstellung, was zu Rundungsfehlern bei Dezimalbruchzahlen führen kann.

- **DECIMAL**(p, s): Exakter, fester Dezimal-Datentyp, ideal für Geldbeträge und präzise Berechnungen.
  - **Precision (p)**: Gesamtanzahl der signifikanten Stellen (Maximalwert z. B. DECIMAL(10,2) → bis 99999999.99).
  - **Scale (s)**: Anzahl der Nachkommastellen (z. B. 2 in DECIMAL(10,2)).
  - **Vorteil**: Speichert Werte als Dezimalbruch, ohne binäre Rundungsfehler, garantiert genaue Rechenergebnisse.

### Zeichenketten-Datentypen
- **CHAR**(n): Feste Länge n, blank-gefüllt, ideal für Codes oder fixe Textlängen.
- **VARCHAR**(n): Variable Länge bis n, speichert nur tatsächliche Zeichenlänge + 1–2 Bytes Overhead.
- **TEXT**-Typen: TINYTEXT (bis 255 Bytes), TEXT (bis 65 535 Bytes), MEDIUMTEXT (bis 16 MiB), LONGTEXT (bis 4 GiB).

### Datums- und Zeittypen
- **DATE**: YYYY-MM-DD, 3 Bytes (z. B. '2025-04-24').
- **TIME**: HH:MM:SS, 3 Bytes, ideal für Tageszeiten ohne Datum.
- **DATETIME**: YYYY-MM-DD HH:MM:SS, 8 Bytes, speichert Datum und Uhrzeit.
- **TIMESTAMP**: 4 Bytes, speichert Zeitstempel seit '1970-01-01 00:00:01' UTC, kann Zeitzone berücksichtigen.
- **YEAR**: 1 Byte, Jahrwert z. B. zwischen 1901 und 2155.

### Boolescher Datentyp
- **BOOLEAN**: In vielen Systemen als TINYINT(1) realisiert, 0 = FALSE, 1 = TRUE.

### Binäre Datentypen
- **BLOB**-Typen: TINYBLOB (bis 255 Bytes), BLOB (bis 65 535 Bytes), MEDIUMBLOB (bis 16 MiB), LONGBLOB (bis 4 GiB), für Binärdateien wie Bilder oder Dokumente.

## Wichtige Einschränkungen (Constraints) in SQL

- **PRIMARY KEY**
  Ein eindeutiger Bezeichner für jede Zeile in einer Tabelle.
  - Gewährleistet, dass jede Zeile eindeutig identifiziert wird (`UNIQUE`).
  - Erlaubt keine `NULL`-Werte (`NOT NULL`).
  - Oft automatisch als Index genutzt und kann z. B. mit `AUTO_INCREMENT` kombiniert werden, um fortlaufende IDs zu erzeugen.

- **UNIQUE**
  Stellt sicher, dass alle Werte in einer oder mehreren Spalten innerhalb der Tabelle einzigartig sind.
  - Kann auf einzelne oder mehrere Spalten angewendet werden (mehrspaltiger Unique-Constraint).
  - Nützlich, um Doppelungen zu vermeiden (z. B. eindeutige E-Mail-Adressen, Benutzernamen).

- **FOREIGN KEY**
  Definiert eine Spalte (oder mehrere), die auf den `PRIMARY KEY` einer anderen Tabelle verweist.
  - Sichert die referenzielle Integrität zwischen Tabellen.
  - Ermöglicht Aktionen bei Löschung oder Aktualisierung von referenzierten Datensätzen (z. B. `ON DELETE CASCADE`, `ON UPDATE RESTRICT`).

- **NOT NULL**
  Verhindert, dass eine Spalte `NULL`-Werte annimmt.
  - Stellt sicher, dass verpflichtende Felder immer einen Wert enthalten (z. B. Geburtsdatum, Vorname).

- **CHECK**
  Legt eine benutzerdefinierte Bedingung fest, die jeder eingetragene Wert erfüllen muss.
  - Beispiel: `CHECK (Alter >= 18)` lässt nur Werte für Erwachsene zu.
  - Hilfreich, um Geschäftslogik direkt auf Datenebene zu validieren.

- **DEFAULT**
  Definiert einen Standardwert für eine Spalte, wenn beim Einfügen kein Wert angegeben wird.
  - Beispiel: `DEFAULT CURRENT_TIMESTAMP` für Zeitstempel-Spalten.
  - Spart wiederholte Angabe häufig genutzter Werte und sorgt für Konsistenz.

In [ ]:
import mysql.connector as mc

try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345") # falsche Password

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

### Erstellen des Cursors
Hier wird ein Cursor mit den Namen "**c**" erstellt. Danach werden alle Tabellen, auf die der Benutzer Berechtigungen besitzt aufgelistet.

In [ ]:
c = conn.cursor()

### Erstellen einer neuen Datenbank auf dem MySQL-Server

In [ ]:
try:
    c.execute("CREATE DATABASE IF NOT EXISTS telefonbuch")
except mc.Error as e:
    print("Error creating database:\n", e)

### Erstellen einer neuen Tabelle
Mithilfe des Befehls "**CREATE TABLE**" wird eine neue Tabelle erstellt:

In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)


# Prüfen ob (noch) eine Verbindung besteht:
if conn.is_connected():
    print("Connection established successfully.")
else:
    print("Connection failed.")

# Cursor erstellen:
c = conn.cursor()

# Neue Tabelle erstellen:

sql = """CREATE TABLE IF NOT EXISTS eintraege (
      id INT(6) UNSIGNED AUTO_INCREMENT PRIMARY KEY,
      vorname VARCHAR(30) NOT NULL,
      nachname VARCHAR(30) NOT NULL,
      vorwahl VARCHAR (10) NOT NULL,
      telefonnummer VARCHAR (30) NOT NULL
      )"""

c.execute(sql)
conn.close()


### Vorhandene Tabellen anzeigen

In [ ]:
c.execute("SHOW TABLES")
for table in c:
    print(table)

Anstatt "**CREATE TABLE IF NOT EXISTS telefonbuch**" kann auch "**CREATE TABLE telefonbuch**" verwendet werden. Durch "**IF NOT EXISTS**" wird aber eine Fehlermeldung vermieden, sollte die Tabelle bereits existieren.  

### Werte hinzufügen
Jetzt fügen wir einen Eintrag hinzu:

In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

# Cursor erstellen:
c = conn.cursor()

var_vorname = input("Vorname: ")
var_nachname = input("Nachname: ")
var_vorwahl = input("Vorwahl: ")
var_telefonnummer = input("Telefonnummer: ")

params = (var_vorname, var_nachname, var_vorwahl, var_telefonnummer)
sql = """INSERT INTO eintraege (vorname, nachname, vorwahl, telefonnummer) VALUES (%s, %s, %s, %s)"""

try:
    c.execute(sql, params)
    conn.commit()

except Exception as e:
    print(e)

finally:
    # Prüfen ob (noch) eine Verbindung besteht:
    if conn and conn.is_connected():
        print("Verbindung zur Datenbank wird getrennt.")
        conn.close()

In Python's **mysql-connector-python** Bibliothek wird **%s** als Platzhalter in SQL-Statements verwendet. Es dient dazu, Werte in das SQL-Statement einzufügen, ohne das Risiko einer SQL-Injection zu erhöhen. Der Platzhalter %s wird unabhängig vom Datentyp des einzufügenden Wertes verwendet. Das bedeutet, **%s wird sowohl für Strings, Zahlen, Datumsangaben als auch andere Datentypen genutzt**.

Es ist wichtig zu beachten, dass der Platzhalter %s in diesem Kontext nichts mit dem String-Formatierungs-Operator in Python zu tun hat. Im Kontext des MySQL-Connectors ist %s einfach ein generischer Platzhalter für Parameter.

Wenn Sie ein SQL-Statement vorbereiten und %s als Platzhalter verwenden, sollten Sie die tatsächlichen Werte als Tupel (oder Liste) an die execute()-Methode übergeben. Der MySQL-Connector sorgt dann dafür, dass diese Werte sicher in das SQL-Statement eingebunden werden, um SQL-Injection zu vermeiden.

Es gibt in diesem Zusammenhang keine anderen Platzhaltertypen wie %d oder %f, die man aus der normalen String-Formatierung in Python kennt. %s wird universell für alle Datentypen verwendet.

#### Mehrere Einträge hinzufügen
Statt **c.execute()** wird **c.executemany()** verwendet. Das Übergeben von Parametern ist nicht erforderlich:

In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

# Cursor erstellen:
c = conn.cursor()

neue_eintraege = [
        ("Max", "Mustermann", "0123", "456789"),
        ("Erika", "Musterfrau", "098", "7654321"),
        ("Hans", "Meier", "0171", "1234567"),
    ]


sql = """INSERT INTO eintraege (vorname, nachname, vorwahl, telefonnummer) VALUES (%s, %s, %s, %s)"""


try:
    c.executemany(sql, neue_eintraege)
    conn.commit()

except Exception as e:
    print(e)

finally:
    # Prüfen ob (noch) eine Verbindung besteht:
    if conn and conn.is_connected():
        print("Verbindung zur Datenbank wird getrennt.")
        conn.close()

### Datenbank auslesen
Jetzt schauen wir nach, welche Einträge in der Tabelle vorhanden sind:


In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

# Cursor erstellen:
c = conn.cursor()

neue_eintraege = [
        ("Max", "Mustermann", "0123", "456789"),
        ("Erika", "Musterfrau", "098", "7654321"),
        ("Hans", "Meier", "0171", "1234567"),
    ]


sql = """INSERT INTO eintraege (vorname, nachname, vorwahl, telefonnummer) VALUES (%s, %s, %s, %s)"""


try:
    c.executemany(sql, neue_eintraege)
    conn.commit()

except Exception as e:
    print(e)

finally:
    # Prüfen ob (noch) eine Verbindung besteht:
    if conn and conn.is_connected():
        print("Verbindung zur Datenbank wird getrennt.")
        conn.close()

### Daten aktualisieren

Die Rufnummer eines Eintrags im Telefonbuch hat sich geändert. Diese müssen wir anpassen:

In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

# Cursor erstellen:
c = conn.cursor()

var_vorname = "Hans"
var_nachname = "Meier"
var_nachname_neu = "Müller"

params = (var_nachname_neu, var_nachname, var_vorname)  # Reihenfolge der Parameter beachten!

sql = """UPDATE eintraege SET nachname = %s WHERE nachname = %s AND vorname = %s"""


try:
    c.execute(sql, params)
    conn.commit()

except Exception as e:
    print(e)

finally:
    # Prüfen ob (noch) eine Verbindung besteht:
    if conn and conn.is_connected():
        print("Verbindung zur Datenbank wird getrennt.")
        conn.close()


### Wir löschen einen Eintrag. 

Als Erstes stellen wir sicher, dass wir den richtigen Benutzer löschen. Dafür brauchen wir am besten eine Angabe, die eindeutig ist.

Da wir vermeiden wollen, Code mehrfach zu schreiben suchen wir uns einen Eintrag aus, indem wir die Datenbank-Abfrage-Zelle ggf. erneut ausführen, und suchen uns die **id** des Eintrags, den wir löschen wollen.


In [ ]:
import mysql.connector as mc

conn = None

# Verbindung herstellen:
try:
    conn = mc.connect(host = "localhost",
                      user = "root",
                      password = "12345",
                      database = "telefonbuch",
                      use_pure = True)

except mc.Error as e:
    print("Error connecting to MySQL Server:\n", e)

# Cursor erstellen:
c = conn.cursor()

var_id = 2

params = (var_id, )  # Reihenfolge der Parameter beachten!

sql = """DELETE FROM eintraege WHERE id = %s"""


try:
    c.execute(sql, params)
    conn.commit()

except Exception as e:
    print(e)

finally:
    # Prüfen ob (noch) eine Verbindung besteht:
    if conn and conn.is_connected():
        print("Verbindung zur Datenbank wird getrennt.")
        conn.close()


In der Zelle "**Datenbank auslesen**" können wir uns vergewissern, ob der Vorgang erfolgreich war.

### Eine keine Aufgabe:
Bitte erstellen Sie einen kleinen Warenkatalog. in diesen Katalog legen Sie bitte Artikel mit den folgenden Merkmalen an:

- ID (primary_key)
- Produkt Name
- Artikelnummer
- Netto-Preis

Schreiben Sie bitte erstmal nur die Erstellung der Datenbank "**warenkatalog**", das Anlegen der benötigten Tabelle und das Hinzufügen der Artikel. Weitere Funktionen können Sie heute Nachmittag in der Übungszeit programmieren.
 


In [15]:
import mysql.connector as mc
from mysql.connector import Error

config = {
    "host": "localhost",      # mejor que "localhost" a veces
    "user": "root",
    "password": "FCbarcelona2025 !",
    "database": "Warenkataloge",
    "use_pure": True,
}

DB_NAME = "warenkataloge"

neue_eintraege = [
    ("Jabon", "AS0123", 7.50),
    ("Papel", "AS0980", 6.25),
    ("Auto BMW",  "AU0171", 1500.00),
    ("Toalla",  "AS0172", 5.00),
]



# Verbindung herstellen:
try:
    conn = mc.connect(**config)
    print("✅ Conectado a MySQL:", conn.get_server_info())

    c = conn.cursor()

    # 1) Crear DB si no existe
    c.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    c.execute(f"USE {DB_NAME}")

    # (Opcional) asegurar tabla existe (evita dudas)
    c.execute("""
    CREATE TABLE IF NOT EXISTS inventario (
        id INT PRIMARY KEY AUTO_INCREMENT,
        produkt_name VARCHAR(100) NOT NULL,
        artikelnummer VARCHAR(50) UNIQUE NOT NULL,
        netto_preis DECIMAL(10,2) NOT NULL
    )
    """)

    # 3) Insertar

    sql = """
    INSERT INTO inventario (produkt_name, artikelnummer, netto_preis)
    VALUES (%s, %s, %s) AS new
    ON DUPLICATE KEY UPDATE
        produkt_name = new.produkt_name,
        netto_preis  = new.netto_preis
    """

    c.executemany(sql, neue_eintraege)

    conn.commit()
    print(f"✅ Insertados: {c.rowcount}")

except Error as e:
    print("❌ MySQL Error:", e)

finally:
    if conn and conn.is_connected():
        conn.close()
        print("🔌 Conexión cerrada.")

✅ Conectado a MySQL: 8.0.45
✅ Insertados: 0
🔌 Conexión cerrada.


Dirks

In [ ]:
import mysql.connector as mc
import mysql


def create_connection():
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="12345",
        db="warenkatalog",
        use_pure=True
    )

    return conn


def create_new_database(db_name):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="12345",
        use_pure=True)

    cursor = conn.cursor()
    try:
        cursor.execute(f"CREATE DATABASE {db_name}")
        print(f"Datenbank '{db_name}' erfolgreich erstellt.")
    except mysql.connector.Error as err:
        print(f"Fehler beim Erstellen der Datenbank: {err}")
    finally:
        conn.close()


def create_products_table():
    conn = create_connection()
    cursor = conn.cursor()
    cursor.execute("""
                   CREATE TABLE IF NOT EXISTS products
                   (
                       id             INT AUTO_INCREMENT PRIMARY KEY,
                       product_name   VARCHAR(255),
                       article_number VARCHAR(255) UNIQUE NOT NULL,
                       net_price      DECIMAL(10, 2)
                   )
                   """)
    conn.commit()
    conn.close()


def add_product(product_name, article_number, net_price):
    conn = create_connection()
    cursor = conn.cursor()
    sql = "INSERT INTO products (product_name, article_number, net_price) VALUES (%s, %s, %s)"
    val = (product_name, article_number, net_price)
    cursor.execute(sql, val)
    conn.commit()
    conn.close()

# Datenbank erstellen:
# create_new_database("warenkatalog")

# Tabelle erstellen:
# create_products_table()

# Produkt hinzufügen:
# add_product("Schreibtisch Stuhl", "0816", 49.95)

## Eine weitere Spalte der Tabelle hinzufügen

Pläne ändern sich. Vielleicht müssen Sie ihre bereits erstellte Tabelle um eine Spalte erweitern.

Um eine vorhandene Tabelle in einer MySQL-Datenbank um eine weitere Spalte zu erweitern, verwenden Sie das SQL-Statement ALTER TABLE. In Python können Sie dieses Statement mit einem MySQL-Connector ausführen.

Dafür gehen Sie so vor:

In [26]:
import mysql.connector as mc

def create_connection():
    return mc.connect(
        host = "localhost",
        user = "root",
        password = "FCbarcelona2025 !",
        database = "Warenkataloge",
        use_pure = True,
    )

def create_stock_col():
    conn = create_connection()
    cursor = conn.cursor()

    try:
        cursor.execute("SELECT DATABASE()")
        print("DB actual:", cursor.fetchone()[0])

        cursor.execute("""
            ALTER TABLE inventario
            ADD COLUMN ort VARCHAR(100) NOT NULL DEFAULT 'Leipzig'
        """)
        conn.commit()
        print("🆕 Columna ort creada")

    except mc.Error as e:
        if e.errno == 1060:
            print("ℹ️ columna ya existía")
        else:
            raise

    finally:
        cursor.execute("SHOW COLUMNS FROM inventario")
        print("Columnas:", [r[0] for r in cursor.fetchall()])
        cursor.close()
        conn.close()

create_stock_col()

DB actual: warenkataloge
🆕 Columna ort creada
Columnas: ['id', 'produkt_name', 'artikelnummer', 'netto_preis', 'stock', 'ort']


Dirks

In [ ]:
def create_stock_col():
    conn = create_connection()
    cursor = conn.cursor()

    table_name = "products"
    new_col_name = "stock"
    col_type = "int(6)"

    sql = f"""ALTER TABLE {table_name} ADD COLUMN {new_col_name} {col_type}"""

    cursor.execute(sql)

## Wildcards

Nicht immer wissen wir genau, wie ein Datensatz genau heißt. In solchen Fällen sind **Wildcards** das Mittel der Wahl.

Die Verwendung von Wildcards in SQL ist eine effektive Methode, um nach Daten zu suchen, wenn der genaue Name oder Wert nicht bekannt ist. In SQL wird häufig der Wildcard-Charakter **%** in Kombination mit dem **LIKE**-Operator verwendet, um teilweise Übereinstimmungen in einer Zeichenfolge zu finden. 

Damit wir ein wenig ausprobieren können, müssen wir noch ein paar Produkte in unserer Warendatenbank anlegen. Vielleicht eine gewisse Anzahl verschiedene Stühle und Tische. :-)

<img style="float: center;" src="img/datensatz.png"><br>

#### Suche nach einem bestimmten Muster am Anfang:


Um den Code ein wenig zu kürzen, erstellen wir eine Verbindungs-Funktion:

In [ ]:
# Ihr Code HIER!

Wenn Sie nach Einträgen suchen möchten, die mit einem bestimmten Muster beginnen, setzen Sie die %-Wildcard **nach** dem Muster.

In [ ]:
SELECT * FROM products WHERE product_name LIKE "Stuhl%"

Wenn Sie nach Einträgen suchen möchten, die mit einem bestimmten Muster beginnen, setzen Sie die %-Wildcard **vor** dem Muster.

In [ ]:
SELECT * FROM products WHERE product_name LIKE "%stuhl"

Wenn Sie nach Einträgen suchen möchten, die ein bestimmtes Muster irgendwo enthalten, setzen Sie die %-Wildcard **vor und nach** dem Muster.

In [ ]:
SELECT * FROM products WHERE product_name LIKE "%Stuhl%"

## Datapipelines

#### Eine Sache noch zum Schluss

Data Pipelines im Kontext von SQL und Datenbanken beziehen sich im Allgemeinen auf **automatisierte Prozesse**, die für die systematische Übertragung und Transformation von Daten von einer Quelle zu einem Ziel verwendet werden. Diese Pipelines spielen eine entscheidende Rolle in der Datenverarbeitung und Business Intelligence. Sie ermöglichen es, große Mengen von Daten effizient und zuverlässig zu bewegen, zu verarbeiten und für Analysezwecke nutzbar zu machen.

Hier sind einige Kernaspekte von Data Pipelines:

- **Datenextraktion (Extraction)**: Dies ist der erste Schritt, bei dem Daten aus einer oder mehreren Quellen extrahiert werden. Diese Quellen können Datenbanken, Dateien, APIs, externe Dienste und andere sein.

- **Datenverarbeitung/-transformation (Transformation)**: Nach der Extraktion werden die Daten häufig transformiert. Diese Transformationen können das Bereinigen von Daten, das Anwenden von Geschäftslogik, das Aggregieren, das Sortieren, das Joinen mit anderen Datensätzen usw. umfassen. Ziel ist es, die Daten in ein Format zu bringen, das für die Analyse oder den nächsten Schritt in der Pipeline geeignet ist.

- **Datenspeicherung (Loading)**: Im letzten Schritt werden die verarbeiteten Daten in einem Zielsystem gespeichert, das ein Data Warehouse, eine Datenbank oder ein anderes Speichersystem sein kann. Dieser Prozess wird oft als ETL-Prozess (Extraction, Transformation, Loading) bezeichnet.

- **Automatisierung**: Data Pipelines sind oft automatisiert, um kontinuierlich oder zu geplanten Zeiten zu laufen. Dies ist besonders wichtig, um die Aktualität der Daten in schnelllebigen Geschäftsumgebungen zu gewährleisten.

- **Überwachung und Wartung**: Da Data Pipelines kritische Geschäftsprozesse unterstützen können, ist ihre ständige Überwachung und regelmäßige Wartung wichtig, um Probleme wie Datenverlust, Verzögerungen oder Fehler in der Datenverarbeitung zu verhindern.

- **SQL in Data Pipelines**: SQL spielt eine entscheidende Rolle in vielen Phasen einer Data Pipeline, insbesondere bei der Extraktion von Daten aus relationalen Datenbanken und bei deren Transformation. SQL wird wegen seiner Effizienz und Effektivität bei der Datenmanipulation und -abfrage in solchen Szenarien häufig eingesetzt.

Data Pipelines sind ein wesentlicher Bestandteil von Data-Warehousing-Lösungen, Business-Intelligence-Systemen, Data-Lake-Implementierungen und anderen datenintensiven Anwendungen, die auf der Analyse und dem Reporting von Daten basieren.

<img style="float: center;" src="img/wbs-logo.jpg"><br>
Autor: Dirk Maric

### Abbildungs- und Quellenverzeichnis

Das Python Logo ist ein eingetragenes Warenzeichen der Python Software Foundation
Alle auf dieser Website veröffentlichten Logos sowie Marken-, Produkt- und Warenzeichen sind Eigentum der jeweiligen Unternehmen
© WBS TRAINING AG – Alle Rechte vorbehalten

### Nutzungsrechte:
Die Nutzung dieser Dokumentation ist ausschließlich für Schulungszwecke der WBS TRAINING AG gestattet. Eine Weitergabe an Dritte, auch auszugsweise, sowie Vervielfältigungen und Verbreitungen aller Art (elektronische und andere Verfahren) inklusive Übersetzungen sind nur mit vorheriger schriftlicher Zustimmung des Rechtinhabers gestattet. Zuwiderhandlungen verpflichten zu Schadenersatz.

### Herausgeber:

WBS TRAINING AG
Lorenzweg 5
12099 Berlin
Haftungsausschluss:
Alle Inhalte sind nach bestem Wissen korrekt und vollständig recherchiert und mit größtmöglicher Sorgfalt für die Schulungsunterlage zusammengestellt. Wir sind um die laufende Aktualisierung aller Informationen und Daten bemüht. Dennoch können Fehler (z.B. Abweichungen zur beschriebenen Hard- und Software durch kurzfristige Updates) auftreten, sodass wir für die vollständige Übereinstimmung, Richtigkeit und Aktualität keine Gewähr übernehmen. Hinweise unserer Nutzer werden konsequent weiterverfolgt.
